# Categorical String Processing

## Ground Truth Checking

This code is designed to evalute the ground truth labels, ensuring they correctly map to documents, they are complete (e.g., no "Unchecked" or empty binary values), and no leading white spaces


In [ ]:
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Tuple


ARTICLE_PDF_PATTERN = re.compile(r"^Article_(\d+)\.pdf$")


def iter_json_files(gt_dir: str) -> Iterable[str]:
    IGNORE = {"Template6_11_2025.json", "Template6_10_2025.json"}

    for root, _, files in os.walk(gt_dir):
        for fn in files:
            if not fn.lower().endswith(".json"):
                continue
            if fn in IGNORE:
                continue
            yield os.path.join(root, fn)


def load_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def contains_unchecked_anywhere(obj: Any) -> bool:
    try:
        s = json.dumps(obj, ensure_ascii=False)
    except TypeError:
        s = str(obj)
    return "unchecked" in s.lower()


def find_invalid_binary_keys(obj: Any) -> List[Tuple[str, Any]]:
    invalid: List[Tuple[str, Any]] = []

    def walk(node: Any, path: str) -> None:
        if isinstance(node, dict):
            for k, v in node.items():
                key_path = f"{path}.{k}" if path else str(k)
                if isinstance(k, str) and k.endswith("_Binary"):
                    if not (isinstance(v, bool) or v == "True" or v == "False"):
                        invalid.append((key_path, v))
                walk(v, key_path)
        elif isinstance(node, list):
            for i, v in enumerate(node):
                walk(v, f"{path}[{i}]")

    walk(obj, "")
    return invalid


def find_strings_starting_with_space(obj: Any) -> List[Tuple[str, str]]:
    bad: List[Tuple[str, str]] = []

    def walk(node: Any, path: str) -> None:
        if isinstance(node, dict):
            for k, v in node.items():
                key_path = f"{path}.{k}" if path else str(k)
                if isinstance(v, str) and v[:1].isspace():
                    bad.append((key_path, v))
                walk(v, key_path)
        elif isinstance(node, list):
            for i, v in enumerate(node):
                walk(v, f"{path}[{i}]")
        else:
            if isinstance(node, str) and node[:1].isspace():
                bad.append((path, node))

    walk(obj, "")
    return bad


@dataclass
class FileEvalResult:
    json_path: str
    json_base: str
    filename_in_json: str
    expected_pdf: str
    check1_article_pattern_ok: bool
    check2_stem_matches_ok: bool
    check3_no_unchecked_ok: bool
    check4_binary_values_ok: bool
    check5_no_leading_space_strings_ok: bool
    invalid_binary_entries: List[Tuple[str, Any]]
    leading_space_string_entries: List[Tuple[str, str]]
    read_error: str | None = None


def evaluate_file(json_path: str, data: Dict[str, Any]) -> FileEvalResult:
    json_base = os.path.basename(json_path)
    stem, _ = os.path.splitext(json_base)
    expected_pdf = f"{stem}.pdf"

    filename_value = data.get("filename", "")
    filename_str = filename_value if isinstance(filename_value, str) else ""

    c1 = bool(ARTICLE_PDF_PATTERN.match(filename_str))
    c2 = (filename_str == expected_pdf)
    c3 = not contains_unchecked_anywhere(data)

    invalid_binaries = find_invalid_binary_keys(data)
    c4 = (len(invalid_binaries) == 0)

    leading_space_strings = find_strings_starting_with_space(data)
    c5 = (len(leading_space_strings) == 0)

    return FileEvalResult(
        json_path=json_path,
        json_base=json_base,
        filename_in_json=filename_str,
        expected_pdf=expected_pdf,
        check1_article_pattern_ok=c1,
        check2_stem_matches_ok=c2,
        check3_no_unchecked_ok=c3,
        check4_binary_values_ok=c4,
        check5_no_leading_space_strings_ok=c5,
        invalid_binary_entries=invalid_binaries,
        leading_space_string_entries=leading_space_strings,
        read_error=None,
    )


def evaluate_gt_dir(gt_dir: str) -> Dict[str, Any]:
    json_files = sorted(iter_json_files(gt_dir))
    all_results: List[FileEvalResult] = []

    failed_1: List[str] = []
    failed_2: List[str] = []
    failed_3: List[str] = []
    failed_4: List[str] = []
    failed_5: List[str] = []
    failed_any: set[str] = set()
    read_errors: List[str] = []

    for path in json_files:
        json_base = os.path.basename(path)
        try:
            data = load_json(path)
            r = evaluate_file(path, data)
        except Exception as e:
            msg = f"{json_base} ({type(e).__name__}: {e})"
            read_errors.append(msg)
            failed_any.add(json_base)
            r = FileEvalResult(
                json_path=path,
                json_base=json_base,
                filename_in_json="",
                expected_pdf=f"{os.path.splitext(json_base)[0]}.pdf",
                check1_article_pattern_ok=False,
                check2_stem_matches_ok=False,
                check3_no_unchecked_ok=False,
                check4_binary_values_ok=False,
                check5_no_leading_space_strings_ok=False,
                invalid_binary_entries=[],
                leading_space_string_entries=[],
                read_error=msg,
            )

        all_results.append(r)

        if r.read_error:
            continue

        if not r.check1_article_pattern_ok:
            failed_1.append(json_base)
            failed_any.add(json_base)

        if not r.check2_stem_matches_ok:
            failed_2.append(json_base)
            failed_any.add(json_base)

        if not r.check3_no_unchecked_ok:
            failed_3.append(json_base)
            failed_any.add(json_base)

        if not r.check4_binary_values_ok:
            failed_4.append(json_base)
            failed_any.add(json_base)

        if not r.check5_no_leading_space_strings_ok:
            failed_5.append(json_base)
            failed_any.add(json_base)

    return {
        "gt_dir": gt_dir,
        "n_files": len(json_files),
        "all_results": all_results,
        "failed_1": failed_1,
        "failed_2": failed_2,
        "failed_3": failed_3,
        "failed_4": failed_4,
        "failed_5": failed_5,
        "unique_failed": sorted(failed_any),
        "read_errors": read_errors,
    }


def print_report(
    results: Dict[str, Any],
    show_invalid_binary_details: bool = False,
    show_leading_space_string_details: bool = False,
) -> None:
    def _print_failures(title: str, failures: List[str]) -> None:
        print(f"\n{title}")
        print(f"  Failed: {len(failures)}")
        for f in failures:
            print(f"    - {f}")

    print(f"Scanning {results['n_files']} JSON files under: {results['gt_dir']}")

    _print_failures('Check 1) "filename" matches pattern Article_<number>.pdf', results["failed_1"])
    _print_failures('Check 2) "filename" matches JSON stem: <stem>.json -> <stem>.pdf', results["failed_2"])
    _print_failures('Check 3) Does NOT contain "Unchecked" anywhere (case-insensitive)', results["failed_3"])
    _print_failures('Check 4) All "*_Binary" keys have value True/False or "True"/"False"', results["failed_4"])
    _print_failures('Check 5) No string values start with whitespace (leading space/tab/newline)', results["failed_5"])

    if show_invalid_binary_details:
        print("\nInvalid *_Binary details (key-path -> value):")
        any_printed = False
        for r in results["all_results"]:
            if r.read_error:
                continue
            if r.invalid_binary_entries:
                any_printed = True
                print(f"  {r.json_base}:")
                for key_path, value in r.invalid_binary_entries:
                    print(f"    - {key_path} -> {value!r}")
        if not any_printed:
            print("  (none)")

    if show_leading_space_string_details:
        print("\nLeading-whitespace string details (key-path -> value):")
        any_printed = False
        for r in results["all_results"]:
            if r.read_error:
                continue
            if r.leading_space_string_entries:
                any_printed = True
                print(f"  {r.json_base}:")
                for key_path, value in r.leading_space_string_entries:
                    print(f"    - {key_path} -> {value!r}")
        if not any_printed:
            print("  (none)")

    if results["read_errors"]:
        print("\nRead/parse errors (counted as failed_any):")
        print(f"  Errors: {len(results['read_errors'])}")
        for e in results["read_errors"]:
            print(f"    - {e}")

    print("\nUNIQUE FILES THAT FAILED ANY CHECK")
    print(f"  Total unique failures: {len(results['unique_failed'])}")
    for f in results["unique_failed"]:
        print(f"    - {f}")

    print("\nDone.")


GT_DIR = "../HUGO-CS/GroundTruth/"
results = evaluate_gt_dir(GT_DIR)
print_report(results, show_invalid_binary_details=False, show_leading_space_string_details=False)


Scanning 244 JSON files under: ../HUGO-CS/GroundTruth/

Check 1) "filename" matches pattern Article_<number>.pdf
  Failed: 0

Check 2) "filename" matches JSON stem: <stem>.json -> <stem>.pdf
  Failed: 0

Check 3) Does NOT contain "Unchecked" anywhere (case-insensitive)
  Failed: 0

Check 4) All "*_Binary" keys have value True/False or "True"/"False"
  Failed: 0

Check 5) No string values start with whitespace (leading space/tab/newline)
  Failed: 0

UNIQUE FILES THAT FAILED ANY CHECK
  Total unique failures: 0

Done.


## Flagging Unmapped Strigns

This code is designed to identify unmapped categorical strings

In [ ]:
import json
import re
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional


EXP_JSON_PATH = "../HUGO-CS/Dataset/AlternateVersions/Extracted-6-10-25_gt_remapped_swapped_deleted_filled_sorted.json"
REPLACE_JSON_PATH = "../HUGO-CS/prompts/regExReplace2.json"


_WS_RE = re.compile(r"\s+")

def norm_key(s: str) -> str:
    return _WS_RE.sub(" ", s.strip()).lower()

def safe_load_extracted_text(extracted: Any) -> Any:
    if isinstance(extracted, str):
        try:
            return json.loads(extracted)
        except json.JSONDecodeError:
            return {}
    return extracted

def flatten_experiments_from_articles(articles: Any) -> List[Dict[str, Any]]:
    all_exps: List[Dict[str, Any]] = []
    if not isinstance(articles, list):
        return all_exps

    for art in articles:
        if not isinstance(art, dict):
            continue

        extracted = safe_load_extracted_text(art.get("extractedText", {}))

        exps: List[Any] = []
        if isinstance(extracted, dict):
            exps = extracted.get("Experiments", []) or []
        elif isinstance(extracted, list):
            for item in extracted:
                if isinstance(item, dict):
                    exps.extend(item.get("Experiments", []) or [])

        for exp in exps:
            if isinstance(exp, dict):
                all_exps.append(exp)

    return all_exps


def audit_group_by_mapped_value(
    experiments: List[Dict[str, Any]],
    section: str,
    feature: str,
    replace_map: Dict[str, Any],
) -> Dict[str, Any]:
    if not isinstance(replace_map, dict):
        replace_map = {}

    # normalized key -> replacement
    norm_to_repl: Dict[str, Any] = {norm_key(k): v for k, v in replace_map.items()}

    # count raw observed strings
    raw_counts = Counter()
    for exp in experiments:
        if not isinstance(exp, dict):
            continue
        props = exp.get(section, {})
        if not isinstance(props, dict):
            continue
        val = props.get(feature)
        if isinstance(val, str):
            raw_counts[val] += 1

    # group raw values by mapped replacement
    mapped_groups: Dict[str, List[Tuple[str, int]]] = defaultdict(list)
    unmapped: List[Tuple[str, int]] = []

    for raw_val, cnt in raw_counts.most_common():
        k = norm_key(raw_val)
        if k in norm_to_repl:
            repl = norm_to_repl[k]
            repl_key = repr(repl)
            mapped_groups[repl_key].append((raw_val, cnt))
        else:
            unmapped.append((raw_val, cnt))

    # sort groups by total frequency descending
    group_order = sorted(
        mapped_groups.items(),
        key=lambda kv: sum(c for _, c in kv[1]),
        reverse=True
    )

    return {
        "raw_counts": raw_counts,
        "mapped_groups_ordered": group_order,  
        "unmapped": unmapped,                  
        "num_unique_raw": len(raw_counts),
        "num_unique_mapped_raw": sum(len(v) for _, v in group_order),
        "num_unique_unmapped_raw": len(unmapped),
    }

def print_grouped_report(
    title: str,
    report: Dict[str, Any],
    max_mapped_value_chars: int = 220,
) -> None:
    print("=" * 120)
    print(title)
    print("-" * 120)


    print("MAPPED (grouped by mapped-to value)")
    group_order = report["mapped_groups_ordered"]
    if group_order:
        for mapped_value_repr, members in group_order:
            total = sum(c for _, c in members)
            mv = mapped_value_repr
            if len(mv) > max_mapped_value_chars:
                mv = mv[: max_mapped_value_chars - 3] + "..."
            print(f"\n  MAPPED TO: {mv}   [total={total}, unique_raw={len(members)}]")
            for raw_val, cnt in members:
                print(f"    {cnt:>6}  {raw_val!r}  ->  {mv}")
    else:
        print("  (none)")
    print(f"\nTotal unique raw values with mapping: {report['num_unique_mapped_raw']}\n")

    print("UNMAPPED (value counts)")
    unmapped = report["unmapped"]
    if unmapped:
        for raw_val, cnt in unmapped:
            print(f"  {cnt:>6}  {raw_val!r}")
    else:
        print("  (none)")
    print(f"\nTotal unique raw values without mapping: {report['num_unique_unmapped_raw']}")
    print(f"Total unique raw values overall: {report['num_unique_raw']}\n")

def run_grouped_mapping_audit(
    exp_json_path: str,
    replace_json_path: str,
    tasks: List[Tuple[str, str, str]],
) -> Dict[str, Any]:
    with open(exp_json_path, "r", encoding="utf-8") as f:
        articles = json.load(f)

    with open(replace_json_path, "r", encoding="utf-8") as f:
        replace_dict = json.load(f)

    experiments = flatten_experiments_from_articles(articles)
    print(f"Loaded {len(experiments)} experiments.\n")

    results: Dict[str, Any] = {
        "exp_json_path": exp_json_path,
        "replace_json_path": replace_json_path,
        "num_experiments": len(experiments),
        "reports": [],
    }

    for section, feature, mapping_name in tasks:
        replace_map = replace_dict.get(mapping_name, {}) or {}
        rep = audit_group_by_mapped_value(experiments, section, feature, replace_map)
        title = f"{section} -> {feature}  (mapping: {mapping_name})"
        print_grouped_report(title, rep)
        results["reports"].append(
            {
                "section": section,
                "feature": feature,
                "mapping_name": mapping_name,
                "report": rep,
            }
        )

    return results


TASKS = [
    ("preSprayedProperties", "Majority_Powder_Material_Name", "materialNamingConventions"),
    ("preSprayedProperties", "Secondary_Powder_Material_Name", "materialNamingConventions"),
    ("preSprayedProperties", "Tertiary_Powder_Material_Name", "materialNamingConventions"),

    ("preSprayedProperties", "Powder_Production_Method", "powderManufacturing"),

    ("experimentalProperties", "Process_Gas_Type", "gasMapping"),
    ("experimentalProperties", "Tensile_Test_Orientation", "tensileOrientation"),
    ("experimentalProperties", "Spraying_System_Model", "systemNameMapping"),
    ("experimentalProperties", "Deposit_Post_Treatment_Description", "treatmentMapping"),
    ("experimentalProperties", "Standard_for_Tensile_Testing", "tensileStandardMapping"),
]

result = run_grouped_mapping_audit(
    exp_json_path=EXP_JSON_PATH,
    replace_json_path=REPLACE_JSON_PATH,
    tasks=TASKS,
)


Loaded 4383 experiments.

preSprayedProperties -> Majority_Powder_Material_Name  (mapping: materialNamingConventions)
------------------------------------------------------------------------------------------------------------------------
MAPPED (grouped by mapped-to value)

  MAPPED TO: '[V] Pure Cu'   [total=720, unique_raw=50]
       246  'Copper'  ->  '[V] Pure Cu'
       150  'Pure Cu'  ->  '[V] Pure Cu'
       122  'Cu'  ->  '[V] Pure Cu'
        38  'Cu powder'  ->  '[V] Pure Cu'
        24  'Copper powder'  ->  '[V] Pure Cu'
        12  'Spherical Copper Powder'  ->  '[V] Pure Cu'
        11  'Pure Copper'  ->  '[V] Pure Cu'
        10  'Electrolytic copper'  ->  '[V] Pure Cu'
        10  'Commercially pure copper'  ->  '[V] Pure Cu'
         6  'CP Cu powder'  ->  '[V] Pure Cu'
         5  'Pure Cu powder'  ->  '[V] Pure Cu'
         4  'Gas-atomized Cu powder'  ->  '[V] Pure Cu'
         4  'Copper (PMS-1 grade)'  ->  '[V] Pure Cu'
         4  '500A Copper'  ->  '[V] Pure Cu'

# Categorical Mapping Review

In [ ]:
import json
import os
import time
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Set

from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

from openai import OpenAI



EXP_JSON_PATH = "../HUGO-CS/Dataset/AlternateVersions/Extracted-6-10-25_gt_remapped_swapped_deleted_filled_sorted.json"
REPLACE_JSON_PATH = "../HUGO-CS/prompts/regExReplace2.json"


TASKS = [
    ("preSprayedProperties", "Majority_Powder_Material_Name", "materialNamingConventions"),
    ("preSprayedProperties", "Secondary_Powder_Material_Name", "materialNamingConventions"),
    ("preSprayedProperties", "Tertiary_Powder_Material_Name", "materialNamingConventions"),
    ("preSprayedProperties", "Powder_Production_Method", "powderManufacturing"),
    ("experimentalProperties", "Process_Gas_Type", "gasMapping"),
    ("experimentalProperties", "Tensile_Test_Orientation", "tensileOrientation"),
    ("experimentalProperties", "Spraying_System_Model", "systemNameMapping"),
    ("experimentalProperties", "Deposit_Post_Treatment_Description", "treatmentMapping"),
]

OUTPUT_DIR = "reviews"

OPENAI_TOKEN_PATH = (
    "../openaiToken.txt"
)

MODEL = "o4-mini"
REASONING_EFFORT = "high"
MAX_WORKERS = 8
MAX_KEYS_PER_CALL = 250
MAX_RETRIES = 5
BACKOFF_SECONDS = 2.0
PREVIEW_LIMIT = 8

def load_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Any, path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
        f.write("\n")
    os.replace(tmp, path)


def clean_json_fence(text: str) -> str:
    # Handles typical ```json ... ``` wrappers
    t = text.strip()
    if t.startswith("```"):
        t = t.replace("```json", "").replace("```", "").strip()
    return t


def try_parse_json(text: str) -> Optional[Dict[str, Any]]:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


def chunk_list(xs: List[str], n: int) -> List[List[str]]:
    return [xs[i : i + n] for i in range(0, len(xs), n)]


def run_llm_value_group_assessment(
    client: OpenAI,
    canonical_value: str,
    keys_for_value: List[str],
    *,
    mapping_name: str,
    all_canonical_values: List[str],
) -> Dict[str, Any]:
   
    system = (
        "You are a meticulous data QA reviewer for string standardization mappings.\n\n"
        "Context:\n"
        "- We have a many-to-one mapping: many raw strings map to one canonical label (VALUE).\n"
        "- You will be given ONE canonical VALUE and a list of KEYS that currently map to it.\n"
        "- You will also be given the FULL list of canonical VALUES that keys may plausibly map to.\n\n"
        "Task:\n"
        "- Classify each KEY into exactly one of three buckets:\n"
        "  (1) correct: clearly should map to this VALUE\n"
        "  (2) incorrect: clearly should NOT map to this VALUE (it more strongly matches a different canonical VALUE)\n"
        "  (3) unsure: ambiguous / insufficient context / borderline cases\n"
        "When in doubt, put the string in 'unsure'. Be conservative.\n\n"
        "Return ONLY valid JSON with this schema:\n"
        "{\n"
        '  "correct": [<strings>],\n'
        '  "incorrect": [<strings>],\n'
        '  "unsure": [<strings>]\n'
        "}\n\n"
        "Rules:\n"
        "- Do not invent new strings.\n"
        "- Every returned string must be taken verbatim from the provided KEYS list.\n"
        "- Prefer 'unsure' over 'incorrect' unless it is clearly wrong.\n"
    )

    user = (
        f"Mapping name: {mapping_name}\n\n"
        "ALL CANONICAL VALUES (possible targets):\n"
        f"{json.dumps(all_canonical_values, ensure_ascii=False, indent=2)}\n\n"
        f"CANONICAL VALUE UNDER REVIEW:\n{canonical_value}\n\n"
        "KEYS THAT CURRENTLY MAP TO THIS VALUE:\n"
        f"{json.dumps(keys_for_value, ensure_ascii=False, indent=2)}\n"
    )

    last_err: Optional[Exception] = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                reasoning_effort=REASONING_EFFORT,
            )
            raw = resp.choices[0].message.content or ""
            tokens = int(getattr(resp.usage, "total_tokens", 0) or 0)

            cleaned = clean_json_fence(raw)
            parsed = try_parse_json(cleaned)
            if parsed is None:
                return {
                    "correct": [],
                    "incorrect": [],
                    "unsure": keys_for_value[:],
                    "raw": cleaned,
                    "tokens": tokens,
                    "error": "invalid_json_from_model",
                }

            correct = parsed.get("correct", [])
            incorrect = parsed.get("incorrect", [])
            unsure = parsed.get("unsure", [])

            if not isinstance(correct, list) or not isinstance(incorrect, list) or not isinstance(unsure, list):
                return {
                    "correct": [],
                    "incorrect": [],
                    "unsure": keys_for_value[:],
                    "raw": cleaned,
                    "tokens": tokens,
                    "error": "wrong_schema_from_model",
                }

            return {
                "correct": correct,
                "incorrect": incorrect,
                "unsure": unsure,
                "raw": cleaned,
                "tokens": tokens,
            }

        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(BACKOFF_SECONDS * attempt)
                continue
            raise RuntimeError(f"LLM call failed after {MAX_RETRIES} attempts: {e}") from e

    raise RuntimeError(f"Unexpected failure: {last_err}")


def normalize_and_reconcile_buckets(
    keys_in_batch: List[str],
    model_out: Dict[str, Any],
) -> Dict[str, List[str]]:
    
    allowed: Set[str] = set(keys_in_batch)

    def filt(xs: Any) -> List[str]:
        if not isinstance(xs, list):
            return []
        out: List[str] = []
        for x in xs:
            if isinstance(x, str) and x in allowed:
                out.append(x)
        return out

    correct = set(filt(model_out.get("correct")))
    incorrect = set(filt(model_out.get("incorrect")))
    unsure = set(filt(model_out.get("unsure")))

    overlap = (correct & incorrect) | (correct & unsure) | (incorrect & unsure)
    if overlap:
        correct -= overlap
        incorrect -= overlap
        unsure |= overlap

    assigned = correct | incorrect | unsure
    missing = allowed - assigned
    unsure |= missing

    def in_order(s: Set[str]) -> List[str]:
        return [k for k in keys_in_batch if k in s]

    return {
        "correct": in_order(correct),
        "incorrect": in_order(incorrect),
        "unsure": in_order(unsure),
    }


def assess_one_value_group(
    client: OpenAI,
    *,
    mapping_name: str,
    canonical_value: str,
    keys_for_value: List[str],
    all_canonical_values: List[str],
) -> Dict[str, Any]:
    batches = chunk_list(keys_for_value, MAX_KEYS_PER_CALL)

    merged_correct: Set[str] = set()
    merged_incorrect: Set[str] = set()
    merged_unsure: Set[str] = set()
    raw_responses: List[str] = []
    tokens_total = 0
    errors: List[str] = []

    for batch in batches:
        out = run_llm_value_group_assessment(
            client,
            canonical_value,
            batch,
            mapping_name=mapping_name,
            all_canonical_values=all_canonical_values,
        )
        tokens_total += int(out.get("tokens", 0) or 0)
        if "raw" in out:
            raw_responses.append(out["raw"])
        if "error" in out:
            errors.append(out["error"])

        reconciled = normalize_and_reconcile_buckets(batch, out)

        b_correct = set(reconciled["correct"])
        b_incorrect = set(reconciled["incorrect"])
        b_unsure = set(reconciled["unsure"])

        for k in b_correct:
            if k in merged_incorrect or k in merged_unsure:
                merged_unsure.add(k)
                merged_incorrect.discard(k)
            else:
                merged_correct.add(k)

        for k in b_incorrect:
            if k in merged_correct or k in merged_unsure:
                merged_unsure.add(k)
                merged_correct.discard(k)
            else:
                merged_incorrect.add(k)

        for k in b_unsure:
            merged_unsure.add(k)
            merged_correct.discard(k)
            merged_incorrect.discard(k)

    all_keys = set(keys_for_value)
    assigned = merged_correct | merged_incorrect | merged_unsure
    merged_unsure |= (all_keys - assigned)

    def in_order(s: Set[str]) -> List[str]:
        return [k for k in keys_for_value if k in s]

    result = {
        "value": canonical_value,
        "count": len(keys_for_value),
        "correct": in_order(merged_correct),
        "incorrect": in_order(merged_incorrect),
        "unsure": in_order(merged_unsure),
        "tokens": tokens_total,
    }
    if errors:
        result["errors"] = errors
    result["raw"] = raw_responses
    return result


def build_value_groups(mapping: Dict[str, Any]) -> Dict[str, List[str]]:
    groups: Dict[str, List[str]] = {}
    for k, v in mapping.items():
        if k == "":
            continue
        if not isinstance(v, str):
            raise TypeError(f"Expected all mapping values to be strings, but key {k!r} has {type(v).__name__}")
        groups.setdefault(v, []).append(k)

    for v in groups:
        groups[v] = sorted(groups[v], key=lambda s: (s.casefold(), s))
    return groups


def format_preview(label: str, items: List[str], limit: int = PREVIEW_LIMIT) -> str:
    if not items:
        return ""
    shown = items[:limit]
    lines = [f"  {label} preview ({len(items)}):"]
    for s in shown:
        lines.append(f"    - {json.dumps(s, ensure_ascii=False)}")
    if len(items) > limit:
        lines.append(f"    - ... ({len(items) - limit} more)")
    return "\n".join(lines)


_WS_RE = re.compile(r"\s+")

def norm_key(s: str) -> str:
    return _WS_RE.sub(" ", s.strip()).lower()

def safe_load_extracted_text(extracted: Any) -> Any:
    if isinstance(extracted, str):
        try:
            return json.loads(extracted)
        except json.JSONDecodeError:
            return {}
    return extracted

def flatten_experiments_from_articles(articles: Any) -> List[Dict[str, Any]]:
    all_exps: List[Dict[str, Any]] = []
    if not isinstance(articles, list):
        return all_exps

    for art in articles:
        if not isinstance(art, dict):
            continue

        extracted = safe_load_extracted_text(art.get("extractedText", {}))

        exps: List[Any] = []
        if isinstance(extracted, dict):
            exps = extracted.get("Experiments", []) or []
        elif isinstance(extracted, list):
            for item in extracted:
                if isinstance(item, dict):
                    exps.extend(item.get("Experiments", []) or [])

        for exp in exps:
            if isinstance(exp, dict):
                all_exps.append(exp)

    return all_exps


def build_mapping_from_experiments_and_replace_map(
    experiments: List[Dict[str, Any]],
    section: str,
    feature: str,
    replace_map: Dict[str, Any],
) -> Tuple[Dict[str, str], List[str]]:
    """
    Builds a many-to-one mapping {raw_string: canonical_string} for the specific (section, feature),
    and also returns the full list of possible canonical values (as strings) derived from replace_map values.

    Notes:
    - Only observed raw strings that are present in replace_map (after normalization) are included.
    - Canonical values are stringified via repr(...) to ensure they are ALWAYS strings (required by build_value_groups).
    """
    if not isinstance(replace_map, dict):
        replace_map = {}

    norm_to_repl: Dict[str, Any] = {norm_key(k): v for k, v in replace_map.items()}

    mapping: Dict[str, str] = {}
    for exp in experiments:
        if not isinstance(exp, dict):
            continue
        props = exp.get(section, {})
        if not isinstance(props, dict):
            continue
        val = props.get(feature)
        if not isinstance(val, str):
            continue

        nk = norm_key(val)
        if nk in norm_to_repl:
            mapping[val] = repr(norm_to_repl[nk])

    all_canonical_values = sorted({repr(v) for v in replace_map.values()}, key=lambda s: (s.casefold(), s))
    return mapping, all_canonical_values


def main() -> None:
    api_key = Path(OPENAI_TOKEN_PATH).read_text(encoding="utf-8").strip()
    os.environ["OPENAI_API_KEY"] = api_key
    client = OpenAI(api_key=api_key)

    articles = load_json(EXP_JSON_PATH)
    replace_dict = load_json(REPLACE_JSON_PATH)

    experiments = flatten_experiments_from_articles(articles)
    print(f"Loaded {len(experiments)} experiments from {EXP_JSON_PATH}.")

    for section, feature, mapping_key in TASKS:
        replace_map = {}
        if isinstance(replace_dict, dict):
            replace_map = replace_dict.get(mapping_key, {}) or {}

        mapping, all_canonical_values = build_mapping_from_experiments_and_replace_map(
            experiments,
            section,
            feature,
            replace_map,
        )

        mapping_name = f"{mapping_key} | {section} -> {feature}"
        output_path = str(Path(OUTPUT_DIR) / f"{section}__{feature}__{mapping_key}_review.json")

        if not isinstance(mapping, dict):
            raise TypeError(f"Expected mapping to be a dict, got {type(mapping).__name__}")

        groups = build_value_groups(mapping)

        group_items: List[Tuple[str, List[str]]] = sorted(
            groups.items(),
            key=lambda kv: (-len(kv[1]), kv[0].casefold(), kv[0]),
        )

        print(f"\n=== {mapping_name} ===")
        print(f"Loaded {len(mapping)} mapped unique raw entries. Evaluating {len(group_items)} value-groups (excluding empty-string key).")
        print(f"Unique canonical values provided to model: {len(all_canonical_values)}")

        results: List[Dict[str, Any]] = []
        token_total = 0

        jobs = list(enumerate(group_items, start=1))  
        job_iter = iter(jobs)

        def worker(job_idx: int, canonical_value: str, keys_for_value: List[str]) -> Dict[str, Any]:
            out = assess_one_value_group(
                client,
                mapping_name=mapping_name,
                canonical_value=canonical_value,
                keys_for_value=keys_for_value,
                all_canonical_values=all_canonical_values,
            )
            out["job_idx"] = job_idx
            out["num_groups"] = len(group_items)
            return out

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {}
            pending_by_idx: Dict[int, Dict[str, Any]] = {}
            next_to_print = 1

            while len(futures) < MAX_WORKERS:
                try:
                    job_idx, (value, keys_for_value) = next(job_iter)
                except StopIteration:
                    break
                fut = pool.submit(worker, job_idx, value, keys_for_value)
                futures[fut] = job_idx

            while futures:
                done, _ = wait(futures, return_when=FIRST_COMPLETED)
                for fut in done:
                    job_idx = futures.pop(fut)
                    pending_by_idx[job_idx] = fut.result()

                while next_to_print in pending_by_idx:
                    row = pending_by_idx.pop(next_to_print)

                    token_total += int(row.get("tokens", 0) or 0)
                    results.append(row)

                    print(
                        f"[{row['job_idx']}/{row['num_groups']}] "
                        f"value={row['value']!r} "
                        f"count={row['count']} "
                        f"correct={len(row['correct'])} "
                        f"incorrect={len(row['incorrect'])} "
                        f"unsure={len(row['unsure'])} "
                        f"tokens_total={token_total}"
                    )
                    inc_block = format_preview("Incorrect", row.get("incorrect", []))
                    uns_block = format_preview("Unsure", row.get("unsure", []))
                    if inc_block:
                        print(inc_block)
                    if uns_block:
                        print(uns_block)
                    print("-" * 80)

                    next_to_print += 1

                while len(futures) < MAX_WORKERS:
                    try:
                        job_idx, (value, keys_for_value) = next(job_iter)
                    except StopIteration:
                        break
                    fut = pool.submit(worker, job_idx, value, keys_for_value)
                    futures[fut] = job_idx

        results_sorted = sorted(results, key=lambda r: r.get("job_idx", 10**18))

        out_obj = {
            "exp_json_path": EXP_JSON_PATH,
            "replace_json_path": REPLACE_JSON_PATH,
            "section": section,
            "feature": feature,
            "mapping_key": mapping_key,
            "model": MODEL,
            "reasoning_effort": REASONING_EFFORT,
            "total_tokens": token_total,
            "group_count": len(group_items),
            "unique_canonical_values_count": len(all_canonical_values),
            "unique_canonical_values": all_canonical_values,
            "results": results_sorted,
        }
        save_json(out_obj, output_path)
        print(f"Saved {len(results_sorted)} group assessments to {output_path}. Total tokens: {token_total}")


if __name__ == "__main__":
    main()


# Categorical Mapping Proposals

In [ ]:
import json
import os
import time
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

from openai import OpenAI


EXPERIMENT_JSON = "../HUGO-CS/Dataset/AlternateVersions/Extracted-6-10-25_gt_remapped_swapped_deleted_filled_sorted.json"
REPLACE_JSON = "../HUGO-CS/prompts/regExReplace2.json"

TASKS = [
    ("preSprayedProperties", "Majority_Powder_Material_Name", "materialNamingConventions"),
    ("preSprayedProperties", "Secondary_Powder_Material_Name", "materialNamingConventions"),
    ("preSprayedProperties", "Tertiary_Powder_Material_Name", "materialNamingConventions"),
    ("preSprayedProperties", "Powder_Production_Method", "powderManufacturing"),
    ("experimentalProperties", "Process_Gas_Type", "gasMapping"),
    ("experimentalProperties", "Tensile_Test_Orientation", "tensileOrientation"),
    ("experimentalProperties", "Spraying_System_Model", "systemNameMapping"),
    ("experimentalProperties", "Deposit_Post_Treatment_Description", "treatmentMapping"),
]

OUTPUT_DIR = "proposals"

REPLACE_MAP_KEY = "materialNamingConventions"

SECTION = "preSprayedProperties"
PRIMARY_CANDIDATES = ["Majority_Powder_Material_Name", "Primary_Powder_Material_Name"]
SECONDARY_CANDIDATES = ["Second_Majority_Powder_Material_Name", "Secondary_Powder_Material_Name"]
TERTIARY_CANDIDATES = ["Third_Majority_Powder_Material_Name", "Tertiary_Powder_Material_Name"]

OPENAI_TOKEN_PATH = (
    "../openaiToken.txt"
)
MODEL = "o4-mini"
REASONING_EFFORT = "high"
MAX_RETRIES = 5
BACKOFF_SECONDS = 2.0
MAX_UNMAPPED_PER_CALL = 10
EXAMPLES_PER_VALUE = 5


def load_json(path: str) -> Any:
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def try_parse_json_string(s: str) -> Any:
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        return {}


def safe_json_parse(text: str) -> Optional[Any]:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


def clean_json_fence(text: str) -> str:
    t = (text or "").strip()
    if t.startswith("```"):
        t = t.replace("```json", "").replace("```", "").strip()
    return t


def flatten_experiments(articles: Any) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    if not isinstance(articles, list):
        return out

    for art in articles:
        if not isinstance(art, dict):
            continue

        extracted = art.get("extractedText", {})
        if isinstance(extracted, str):
            extracted = try_parse_json_string(extracted)

        exps: List[Any] = []
        if isinstance(extracted, dict):
            exps = extracted.get("Experiments", []) or []
        elif isinstance(extracted, list):
            for item in extracted:
                if isinstance(item, dict):
                    exps.extend(item.get("Experiments", []) or [])
        else:
            exps = []

        for exp in exps:
            if isinstance(exp, dict):
                out.append(exp)

    return out


def pick_feature_name(
    experiments: List[Dict[str, Any]],
    section: str,
    candidates: List[str]
) -> Optional[str]:
    candidate_set = set(candidates)
    for exp in experiments:
        props = exp.get(section)
        if isinstance(props, dict):
            for k in props.keys():
                if k in candidate_set:
                    return k
    return None


def build_case_insensitive_map(replace_map: Any) -> Dict[str, str]:
    out: Dict[str, str] = {}
    if not isinstance(replace_map, dict):
        return out
    for k, v in replace_map.items():
        if isinstance(k, str) and isinstance(v, str):
            out[k.strip().lower()] = v
    return out


def iter_material_strings(
    experiments: List[Dict[str, Any]],
    section: str,
    feature: str
) -> List[str]:
    vals: List[str] = []
    for exp in experiments:
        if not isinstance(exp, dict):
            continue
        props = exp.get(section)
        if not isinstance(props, dict):
            continue
        v = props.get(feature)
        if isinstance(v, str):
            s = v.strip()
            if s:
                vals.append(s)
    return vals


def build_observed_example_map(
    all_strings: List[str],
    lower_map: Dict[str, str]
) -> Tuple[Dict[str, Counter], Counter]:
    canonical_to_raw_counts: Dict[str, Counter] = defaultdict(Counter)
    unmapped_counts: Counter = Counter()

    for raw in all_strings:
        mapped = lower_map.get(raw.lower())
        if mapped is None:
            unmapped_counts[raw] += 1
        else:
            canonical_to_raw_counts[mapped][raw] += 1

    return canonical_to_raw_counts, unmapped_counts


def build_value_examples_block(
    canonical_to_raw_counts: Dict[str, Counter],
    examples_per_value: int = EXAMPLES_PER_VALUE
) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for canonical in sorted(canonical_to_raw_counts.keys(), key=lambda s: (s.casefold(), s)):
        c = canonical_to_raw_counts[canonical]
        if not c:
            continue
        items = sorted(c.items(), key=lambda kv: (-kv[1], kv[0].casefold(), kv[0]))
        out[canonical] = [k for (k, _n) in items[:examples_per_value]]
    return out


def build_material_naming_guidelines_text() -> str:
    return (
        "Material naming guidelines:\n"
        "- Normalize characters: convert subscripts/superscripts and odd dashes/whitespace to plain ASCII (e.g., Cr3C2, Ti-6Al-4V).\n"
        "- Pure elements: standardize to 'Pure XX' (e.g., Pure Al, Pure Cu). Titanium exception: 'Ti Grade X' when grade 1–4 is stated, else 'Pure Ti'.\n"
        "- Remove morphology/shape and processing descriptors (spherical/irregular, gas-atomized, ball-milled, mesh sizes, etc.).\n"
        "- Parameterized mixture families: replace variable numeric ratios with '(X)' (e.g., WC-12Co -> WC-(X)Co; Ni-20Cr -> Ni-(X)Cr; Al-10Sn -> Al-(X)Sn).\n"
        "- Composition-number simplification: often strip numeric coefficients/percentages and keep element combination (e.g., Cu-0.5Cr-0.05Zr -> CuCrZr; Al0.4CoCrFeNi -> AlCoCrFeNi),\n"
        "  except retain numbers when part of community-standard names (e.g., Ti-6Al-4V, Cr3C2, Ti2AlC, Al2O3, B4C).\n"
        "- Aluminum alloys: 'Al XXXX' (e.g., Al 6061, Al A380). Remove temper/vendor/mesh.\n"
        "- Stainless steels: 'XXX SS' (e.g., 316 SS) and preserve low-carbon variants explicitly (e.g., 316L SS).\n"
        "- Ti-6Al-4V family: map Ti6Al4V/TC4/Grade 5 to 'Ti-6Al-4V'; retain 'Ti-6Al-4V (ELI -- Grade 23)' when indicated.\n"
        "- Trade-name alloys: consolidate formatting; keep distinct grades (e.g., IN 718 vs IN 625).\n"
        "- Carbon/nanostructures: consolidate families (MWCNT/MWCNTs -> CNT; graphene nanoplatelets -> Graphene; etc.).\n"
    )


def build_llm_context_prompt(
    canonical_values: List[str],
    canonical_to_examples: Dict[str, List[str]],
) -> str:
    examples_json = json.dumps(canonical_to_examples, ensure_ascii=False, indent=2)
    canon_json = json.dumps(canonical_values, ensure_ascii=False, indent=2)

    return (
        "You are helping standardize cold-spray powder material names into canonical categories.\n\n"
        f"{build_material_naming_guidelines_text()}\n"
        "EXISTING CANONICAL VALUES (choose from these when possible):\n"
        f"{canon_json}\n\n"
        "OBSERVED EXAMPLE RAW STRINGS PER CANONICAL VALUE (<=5 examples each):\n"
        f"{examples_json}\n"
    )


def chunk_list(xs: List[Any], n: int) -> List[List[Any]]:
    return [xs[i:i + n] for i in range(0, len(xs), n)]


def recommend_for_batch(
    client: OpenAI,
    *,
    context_prompt: str,
    canonical_values: List[str],
    batch_items: List[Tuple[str, int]],
) -> Dict[str, str]:
    """
    batch_items: [(unmapped_string, count), ...]
    Returns: {unmapped_string: "[Existing]: <value>" or "[New]: <value>"}
    """
    system = (
        "You are a meticulous data curator.\n"
        "Given unmapped powder material strings, recommend a canonical label.\n\n"
        "Output requirements:\n"
        "- Return ONLY valid JSON.\n"
        "- Schema: {\"recommendations\": [{\"raw\": <string>, \"recommended\": <string>}]}.\n"
        "- Each 'recommended' must be one of:\n"
        "  - \"[Existing]: <CANONICAL_VALUE>\" where CANONICAL_VALUE is EXACTLY one item from the provided canonical list; OR\n"
        "  - \"[New]: <CANONICAL_VALUE>\" where CANONICAL_VALUE is a new canonical label following the style rules.\n"
        "- Do not invent or alter 'raw' strings; they must match the input exactly.\n"
        "- Prefer [Existing] when a good match exists; use [New] only when necessary.\n"
    )

    user = (
        f"{context_prompt}\n\n"
        "UNMAPPED STRINGS (with occurrence counts):\n"
        f"{json.dumps([{'raw': r, 'count': c} for (r, c) in batch_items], ensure_ascii=False, indent=2)}\n"
    )

    last_err: Optional[Exception] = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                reasoning_effort=REASONING_EFFORT,
            )
            raw_text = (resp.choices[0].message.content or "")
            cleaned = clean_json_fence(raw_text)
            parsed = safe_json_parse(cleaned)

            if not isinstance(parsed, dict) or "recommendations" not in parsed:
                raise ValueError("Model returned invalid JSON schema")

            recs = parsed.get("recommendations")
            if not isinstance(recs, list):
                raise ValueError("Model returned non-list recommendations")

            out: Dict[str, str] = {}
            allowed_raw = {r for (r, _c) in batch_items}

            for item in recs:
                if not isinstance(item, dict):
                    continue
                r = item.get("raw")
                rec = item.get("recommended")
                if isinstance(r, str) and r in allowed_raw and isinstance(rec, str):
                    out[r] = rec

            for r, _c in batch_items:
                if r not in out:
                    out[r] = "[New]: Unknown"

            canon_set = set(canonical_values)
            for r in list(out.keys()):
                rec = out[r]
                if rec.startswith("[Existing]:"):
                    cand = rec[len("[Existing]:"):].strip()
                    if cand not in canon_set:
                        out[r] = f"[New]: {cand or 'Unknown'}"

            return out

        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(BACKOFF_SECONDS * attempt)
                continue
            raise RuntimeError(f"LLM recommend batch failed after {MAX_RETRIES} attempts: {e}") from e

    raise RuntimeError(f"Unexpected failure: {last_err}")


def main() -> None:
    articles = load_json(EXPERIMENT_JSON)
    replace_dict = load_json(REPLACE_JSON)

    experiments = flatten_experiments(articles)

    api_key = Path(OPENAI_TOKEN_PATH).read_text(encoding="utf-8").strip()
    os.environ["OPENAI_API_KEY"] = api_key
    client = OpenAI(api_key=api_key)

    for section, feature, mapping_key in TASKS:
        replace_map = replace_dict.get(mapping_key, {})
        lower_map = build_case_insensitive_map(replace_map)

    
        primary_feature = pick_feature_name(experiments, section, [feature]) or feature
        secondary_feature = pick_feature_name(experiments, section, [feature]) or feature
        tertiary_feature = pick_feature_name(experiments, section, [feature]) or feature

        
        primary_vals = iter_material_strings(experiments, section, primary_feature)
        secondary_vals = iter_material_strings(experiments, section, secondary_feature)
        tertiary_vals = iter_material_strings(experiments, section, tertiary_feature)
        all_vals = primary_vals + secondary_vals + tertiary_vals

        canonical_to_raw_counts, unmapped_counts = build_observed_example_map(all_vals, lower_map)
        canonical_values = sorted(set(lower_map.values()), key=lambda s: (s.casefold(), s))
        canonical_to_examples = build_value_examples_block(canonical_to_raw_counts, examples_per_value=EXAMPLES_PER_VALUE)

        context_prompt = build_llm_context_prompt(canonical_values, canonical_to_examples)
        print("\n" + "=" * 120)
        print("LLM CONTEXT PROMPT (copy/paste if desired)")
        print("=" * 120)
        print(f"Task: {section} -> {feature}  (mapping: {mapping_key})")
        print("-" * 120)
        print(context_prompt)
        print("=" * 120 + "\n")

        unmapped_items = sorted(
            unmapped_counts.items(),
            key=lambda kv: (-kv[1], kv[0].casefold(), kv[0]),
        )

        if not unmapped_items:
            print("No unmapped strings found across primary/secondary/tertiary.")
            continue

        print(f"Found {len(unmapped_items)} UNIQUE unmapped strings (primary+secondary+tertiary combined).")

        batches = chunk_list(unmapped_items, MAX_UNMAPPED_PER_CALL)
        recommendations: Dict[str, str] = {}

        for i, batch in enumerate(batches, start=1):
            print(f"LLM recommending batch {i}/{len(batches)} (size={len(batch)})...")
            recs = recommend_for_batch(
                client,
                context_prompt=context_prompt,
                canonical_values=canonical_values,
                batch_items=batch,
            )
            recommendations.update(recs)

        print("\n" + "=" * 120)
        print("RECOMMENDED MAPPINGS FOR UNMAPPED STRINGS (descending frequency)")
        print("=" * 120)

        for raw, count in unmapped_items:
            rec = recommendations.get(raw, "[New]: Unknown")
            print(f"{count:>5}  {raw!r} -> {rec}")

        existing_n = 0
        new_n = 0
        for raw, _count in unmapped_items:
            rec = recommendations.get(raw, "")
            if rec.startswith("[Existing]:"):
                existing_n += 1
            else:
                new_n += 1

        print("\n" + "-" * 120)
        print("SUMMARY")
        print(f"  Unique unmapped strings: {len(unmapped_items)}")
        print(f"  Recommended [Existing]: {existing_n}")
        print(f"  Recommended [New]     : {new_n}")
        print("-" * 120)

        out_path = Path(OUTPUT_DIR) / f"unmapped_{section}__{feature}__{mapping_key}.json"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_obj = {
            "section": section,
            "feature": feature,
            "mapping_key": mapping_key,
            "unmapped_unique_count": len(unmapped_items),
            "unmapped_items": [
                {"raw": raw, "count": count, "recommended": recommendations.get(raw, "[New]: Unknown")}
                for raw, count in unmapped_items
            ],
        }
        out_path.write_text(json.dumps(out_obj, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        print(f"\nSaved recommendations to: {out_path.resolve()}")


if __name__ == "__main__":
    main()


# Storing to Isolated Files

In [ ]:
import json
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, List, Tuple


REPLACE_JSON = "../HUGO-CS/prompts/regExReplace2.json"


TOP_KEY = ""
TOP_VALUE = "[V] Not Reported"


EXPORTS: List[Tuple[str, str]] = [
    ("gasMapping", "carrierGas.json"),
    ("systemNameMapping", "coldSpraySystem.json"),
    ("powderManufacturing", "PowderManufacturingProcess.json"),
    ("treatmentMapping", "DepositionTreatment.json"),
    ("tensileOrientation", "tensileOrientation.json"),
    ("materialNamingConventions", "materialNaming.json"),
    ("elementMapping", "primaryElementMapping.json"),
]

OUTPUT_DIR = Path("finalMappings")  


def normalize_ws(s: str) -> str:
    """Normalize whitespace for consistent grouping/sorting."""
    return " ".join(s.split())


def sort_mapping_by_value_then_key(mapping: Dict[str, Any], *, group_name: str) -> "OrderedDict[str, str]":
    """
    Sort rules:
      1) Alphabetically by value (string), case-insensitive, with whitespace normalized.
      2) Within the same value, alphabetically by key, case-insensitive.
    Strict: requires all values to be strings.
    """
    pairs: List[Tuple[str, str]] = []

    for k, v in mapping.items():
        if k == TOP_KEY:
            continue
        if not isinstance(v, str):
            raise TypeError(
                f"{group_name}: expected all values to be strings, but key {k!r} has {type(v).__name__}"
            )
        pairs.append((k, v))

    pairs_sorted = sorted(
        pairs,
        key=lambda kv: (
            normalize_ws(kv[1]).casefold(),  
            kv[0].casefold(),                
            kv[0],                           
        ),
    )

    out: "OrderedDict[str, str]" = OrderedDict()
    out[TOP_KEY] = TOP_VALUE
    for k, v in pairs_sorted:
        out[k] = v
    return out


def save_json_with_group_spacing(obj: "OrderedDict[str, str]", path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    items = list(obj.items())

    with path.open("w", encoding="utf-8") as f:
        f.write("{\n")

        prev_group = None
        for i, (k, v) in enumerate(items):
            
            if i == 0:
                curr_group = None
            else:
                curr_group = normalize_ws(v).casefold()

            
            if i > 1 and curr_group != prev_group:
                f.write("\n")

            comma = "," if i < len(items) - 1 else ""
            f.write(f"  {json.dumps(k, ensure_ascii=False)}: {json.dumps(v, ensure_ascii=False)}{comma}\n")

            if i >= 1:
                prev_group = curr_group

        f.write("}\n")


def main() -> None:
    in_path = Path(REPLACE_JSON)
    with in_path.open("r", encoding="utf-8") as f:
        replace_dict = json.load(f)

    for src_key, out_name in EXPORTS:
        mapping = replace_dict.get(src_key)
        if mapping is None:
            raise KeyError(f"Missing required group {src_key!r} in {in_path}")
        if not isinstance(mapping, dict):
            raise TypeError(f"Expected {src_key!r} to be a dict, got {type(mapping).__name__}")

        normalized = sort_mapping_by_value_then_key(mapping, group_name=src_key)
        save_json_with_group_spacing(normalized, OUTPUT_DIR / out_name)
        print(f"[WROTE] {src_key} -> {out_name} (items={len(normalized)})")


if __name__ == "__main__":
    main()


[WROTE] gasMapping -> carrierGas.json (items=60)
[WROTE] systemNameMapping -> coldSpraySystem.json (items=765)
[WROTE] powderManufacturing -> PowderManufacturingProcess.json (items=357)
[WROTE] treatmentMapping -> DepositionTreatment.json (items=837)
[WROTE] tensileOrientation -> tensileOrientation.json (items=130)
[WROTE] materialNamingConventions -> materialNaming.json (items=1455)
[WROTE] elementMapping -> primaryElementMapping.json (items=133)
